# CSE 151B — Submission notebook (`private.jsonl`)

Live in **`notebooks/submission.ipynb`**. Same inference stack as [`full_public.ipynb`](full_public.ipynb), run on all of **`data/private.jsonl`** (no labels).

1. (**Colab A100**) `%pip` → restart runtime → Drive copy cell.
2. Set **`MAX_TOKENS`** in §2 (default **32k**, matching pub-003).
3. Prompts are **per question**: baseline for MCQ and single-blank free-form; multi-blank format when `[ANS]` count > 1.
4. Generate with checkpointing → write **`results/submission.csv`**.

**Output:** CSV with columns `id`, `response` (full model trace, CSV-escaped). Checkpoint: `results/submission.responses.jsonl`.

### Google Colab

**Colab (A100):** run the `%pip` cell below, restart runtime, continue. **Local:** use your venv — same packages (`vllm`, `transformers`, …).

> **Note:** `bitsandbytes` is not needed — Qwen3-4B fits in native bf16 on A100 (~8 GB), which is faster than quantized loading.

In [1]:
# Colab: skip when working locally with an existing venv.
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 MB 84.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 81.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 253.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 125.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 257.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 243.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 196.0 MB/s eta 0:00:0000:0100:01
     ━━━━

## 2. Imports & configuration

- `MAX_TOKENS` — generation budget (current: 32768)
- `PROMPT_VARIANT` — names the prompt build defined in §5. Baked into the
  output filename so two runs with different prompts cannot share a checkpoint
- `SUBMISSION_CSV` — auto-derived: `results/submission_{PROMPT_VARIANT}_{N}k.csv`
- `DATA_PATH` — `data/private.jsonl`

The responses checkpoint (`{stem}.responses.jsonl`) inherits the same stem, so
changing `PROMPT_VARIANT` automatically gives a fresh checkpoint — no risk of
resuming from responses generated under an older prompt.


In [2]:
import csv
import json
import os
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()

MAX_TOKENS = 16384

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"
DATA_PATH = str(REPO_ROOT / "data/private.jsonl")

# Identifies the prompt build defined in §5. Must match a variant defined in
# notebooks/dev.ipynb when we want dev/submission to agree. Used to namespace
# SUBMISSION_CSV + the responses checkpoint so a prompt change cannot silently
# resume from responses generated under an older prompt.
PROMPT_VARIANT = "exact_v1"

# vLLM's max_model_len is the TOTAL context (prompt + generation). Reserve
# PROMPT_BUDGET for the precision_xml system (~700 tok) + longest expected
# question + chat-template overhead, so changing MAX_TOKENS automatically grows
# max_model_len. Without this, a higher MAX_TOKENS gets silently clamped to
# (max_model_len - prompt_len).
PROMPT_BUDGET = 2048
MAX_MODEL_LEN = MAX_TOKENS + PROMPT_BUDGET

_TOKEN_K = MAX_TOKENS // 1024
SUBMISSION_CSV = str(REPO_ROOT / f"results/submission_{PROMPT_VARIANT}_{_TOKEN_K}k.csv")

print(f"REPO_ROOT      = {REPO_ROOT}")
print(f"MAX_TOKENS     = {MAX_TOKENS} ({_TOKEN_K}k)   # new-token cap")
print(f"MAX_MODEL_LEN  = {MAX_MODEL_LEN}             # prompt + generation cap")
print(f"PROMPT_VARIANT = {PROMPT_VARIANT}")
print(f"SUBMISSION_CSV = {SUBMISSION_CSV}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import sys
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

REPO_ROOT      = /content
MAX_TOKENS     = 16384 (16k)   # new-token cap
MAX_MODEL_LEN  = 18432             # prompt + generation cap
PROMPT_VARIANT = exact_v1
SUBMISSION_CSV = /content/results/submission_exact_v1_16k.csv


## 3. Colab: copy `private.jsonl` from Drive

Upload **`private.jsonl`** to `My Drive/CSE151B/data/private.jsonl` (or change `DRIVE_PRIVATE`). Skip on local clone when `data/private.jsonl` exists.

In [3]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use repo `data/private.jsonl`.")
else:
    drive.mount("/content/drive")
    DRIVE_PRIVATE = Path("/content/drive/MyDrive/CSE151B/data/private.jsonl")
    DRIVE_PROJECT_ROOT = DRIVE_PRIVATE.parent.parent
    if not DRIVE_PRIVATE.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_PRIVATE}\nUpload private.jsonl or set DRIVE_PRIVATE."
        )
    (REPO_ROOT / "data").mkdir(parents=True, exist_ok=True)
    dest = REPO_ROOT / "data/private.jsonl"
    shutil.copy2(DRIVE_PRIVATE, dest)
    print(f"Copied to {dest.resolve()}")

Mounted at /content/drive
Copied to /content/data/private.jsonl


## 4. Load dataset

Private rows have `question`, optional `options`, and `id` — **no** `answer` field.

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |

In [4]:
def n_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")


data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form) from {DATA_PATH}")

mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))
multi_sample = next(
    (d for d in data if not d.get("options") and n_ans_placeholders(d["question"]) > 1),
    free_sample,
)

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2)[:1200], "...\n" if len(json.dumps(mcq_sample)) > 1200 else "\n")
print("── Free-form sample ──")
print(json.dumps(free_sample, indent=2)[:1200], "...\n" if len(json.dumps(free_sample)) > 1200 else "\n")
if multi_sample is not free_sample:
    print(f"── Multi-blank sample ({n_ans_placeholders(multi_sample['question'])} blanks) ──")
    print(json.dumps(multi_sample, indent=2)[:1200], "...\n" if len(json.dumps(multi_sample)) > 1200 else "\n")

Loaded 943 questions  (300 MCQ, 643 free-form) from /content/data/private.jsonl

── MCQ sample ──
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
} 

── Free-form sample ──
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
} 



## 5. Prompt construction

Variant: **`exact_v1`** (mirrors `notebooks/dev.ipynb` §6). Plain-text system prompt, the post-`priv-002` rewrite of `precision_xml_verify`:

1. **Precision** — prefer exact closed forms; decimals only when unavoidable, at conventional precision, **never digit-padded** (the grader's 1e-8 tolerance rejects `1.6448536270` vs rounded gold `1.645`).
2. **No representation gamble** — simplest standard form; echo the question's True/False / Yes/No vocabulary.
3. **Plain text** — no XML `<rule>`/`<instructions>` scaffolding (24% rule-echo), no verify clause; light "be concise" directive only.

**User prompt** per type: MCQ lists `A. … B. …` options; free-form appends the layout rule. **Multi-blank** (2+ `[ANS]`) emits a **single** `\boxed{a, b, c}` box in `[ANS]` order (grader-identical to separate boxes, more robust).


In [5]:
# ── exact_v1 (mirrors notebooks/dev.ipynb §6) ────────────────────────────────
# Post-priv-002 rewrite of precision_xml_verify. Three fixes + single-box multi-blank:
#   (1) PRECISION  — exact forms preferred; decimals only when unavoidable, at
#                    conventional precision, NEVER digit-padded (the grader's 1e-8
#                    relative tolerance rejects 1.6448536270 vs rounded gold 1.645).
#   (2) NO REP GAMBLE — simplest standard form; echo the question's True/False / Yes/No.
#   (3) PLAIN TEXT — no XML <rule>/<instructions> scaffolding (24% echo) and no verify
#                    clause; keep only a light "be concise" directive.
# Multi-blank emits ONE box \boxed{a, b, c} (grader-identical to separate boxes via
# bracket-aware split_by_comma, and immune to contiguous-group/separator pitfalls).

_ROLE_SOLVE = (
    "You are a precise mathematical reasoner. Solve the problem rigorously, but keep "
    "your reasoning concise: do not re-derive steps you have already completed or go "
    "in circles."
)
_ROLE_MCQ = (
    "You are a precise mathematical reasoner. Read the problem and the answer choices, "
    "then identify the single best answer. Keep your reasoning concise; you may confirm "
    "by elimination."
)

_PRECISION_RULE = (
    "Answer precision is graded against a rounded reference with a very tight tolerance, "
    "so the FORM of your answer matters:\n"
    "- Prefer an EXACT closed form whenever one exists - fractions (5/8), radicals "
    "(3\\sqrt{2}), \\pi, e, logarithms (\\log_3 35). Box it exactly; the grader matches "
    "exact forms reliably and they carry no rounding error.\n"
    "- Use a decimal ONLY when no exact form exists (e.g. a statistical test statistic, "
    "p-value, regression coefficient, or physical measurement). Then use the CONVENTIONAL "
    "precision for the problem: standard statistical constants in their usual rounded form "
    "(z = 1.96, 1.645, 2.576; the usual t-table values), and otherwise 2-4 decimal places, "
    "or the precision of the data you were given.\n"
    "- NEVER pad an answer with extra significant figures. An over-precise decimal such as "
    "1.6448536270 will FAIL against a rounded gold value like 1.645. Do not round a true "
    "exact form to a decimal, and do not over-expand a conventional decimal."
)

_REPRESENTATION_RULE = (
    "Give the answer in the simplest standard form and do not rewrite it into a fancier "
    "equivalent (keep \\log_3 35 rather than \\frac{\\ln 35}{\\ln 3}; keep a clean factored "
    "or simplified expression). For True/False or Yes/No answers, use exactly the words the "
    "question uses - write True / False when it says 'True or False', and Yes / No when it "
    "asks a yes/no question."
)

_FORMAT_RULE = (
    "Box the value only - no units or surrounding words (\\boxed{5}, not \\boxed{5 cm}). "
    "Write a probability as a decimal (0.5, not 50%) and a percentage as a bare number "
    "(50, no % sign). Keep any coordinate, tuple, interval, or set together with its "
    "brackets, e.g. \\boxed{(-1, -3)} or \\boxed{[0, 14]}, so its internal commas are not "
    "mistaken for answer separators."
)

_LAYOUT_SINGLE = (
    "End with a single line 'Final Answer:' followed by exactly one \\boxed{} containing "
    "your answer. Put \\boxed{} nowhere else in the response."
)
_LAYOUT_MCQ = (
    "End with a single line 'Final Answer:' followed by exactly one \\boxed{} containing "
    "ONLY the letter of your chosen option (e.g. \\boxed{C}). If the slot says 'select all "
    "that apply', concatenate the letters alphabetically inside that one box with no "
    "separator (e.g. \\boxed{AB}). Put \\boxed{} nowhere else in the response."
)


def _math_layout_rule(n_blanks: int) -> str:
    if n_blanks > 1:
        return (
            f"This problem has {n_blanks} blanks ([ANS]). End with a single line "
            f"'Final Answer:' followed by exactly ONE \\boxed{{}} that contains all "
            f"{n_blanks} answers, comma-separated, in the order the [ANS] blanks appear, "
            f"e.g. \\boxed{{a, b, c}}. Keep any tuple/interval/set in its own brackets "
            f"inside the box. Use only this one box; put \\boxed{{}} nowhere else."
        )
    return _LAYOUT_SINGLE


SYSTEM_PROMPT_MATH = "\n\n".join([_ROLE_SOLVE, _PRECISION_RULE, _REPRESENTATION_RULE, _FORMAT_RULE])
SYSTEM_PROMPT_MCQ  = "\n\n".join([_ROLE_MCQ, _REPRESENTATION_RULE])


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for the exact_v1 variant."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts}\n\n{_LAYOUT_MCQ}"
    n_blanks = n_ans_placeholders(question)
    return SYSTEM_PROMPT_MATH, f"{question}\n\n{_math_layout_rule(n_blanks)}"


def prompt_mode(question: str, options: Optional[list]) -> str:
    if options:
        return "mcq/exact_v1"
    return "multi-blank/exact_v1" if n_ans_placeholders(question) > 1 else "single-blank/exact_v1"


print("Active variant: exact_v1 (plain text, exact-preferred, single-box multi-blank)")
print(f"\nMath system prompt (first 220 chars):\n  {SYSTEM_PROMPT_MATH[:220]}...")
print(f"\nMCQ  system prompt (first 220 chars):\n  {SYSTEM_PROMPT_MCQ[:220]}...\n")

for label, item in [
    ("MCQ", mcq_sample),
    ("Free-form (single-blank)", free_sample),
    *( [(f"Multi-blank ({n_ans_placeholders(multi_sample['question'])} blanks)", multi_sample)] if multi_sample is not free_sample else [] ),
]:
    mode = prompt_mode(item["question"], item.get("options"))
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"-- {label} [{mode}] --")
    print(f"  user (last 220 chars): ...{usr_p[-220:]}\n")


Active variant: exact_v1 (plain text, exact-preferred, single-box multi-blank)

Math system prompt (first 220 chars):
  You are a precise mathematical reasoner. Solve the problem rigorously, but keep your reasoning concise: do not re-derive steps you have already completed or go in circles.

Answer precision is graded against a rounded re...

MCQ  system prompt (first 220 chars):
  You are a precise mathematical reasoner. Read the problem and the answer choices, then identify the single best answer. Keep your reasoning concise; you may confirm by elimination.

Give the answer in the simplest standa...

-- MCQ [mcq/exact_v1] --
  user (last 220 chars): ...of your chosen option (e.g. \boxed{C}). If the slot says 'select all that apply', concatenate the letters alphabetically inside that one box with no separator (e.g. \boxed{AB}). Put \boxed{} nowhere else in the response.

-- Free-form (single-blank) [multi-blank/exact_v1] --
  user (last 220 chars): ... that contains all 2 answers, com

## 6. Load model with vLLM (A100 profile)

Same **bf16** profile as `full_public.ipynb` — not the starter L4 INT8 path. See [`docs/infra/vllm-inference-config.md`](../docs/infra/vllm-inference-config.md).

| Parameter | Value | Notes |
|-----------|-------|-------|
| `dtype` | `bfloat16` | |
| `max_model_len` | `MAX_TOKENS + PROMPT_BUDGET` | derived in §2; total prompt+generation context |
| `max_num_seqs` | auto-scaled from `max_model_len` | halved as context grows so KV cache fits |
| `max_num_batched_tokens` | `max(max_model_len, 32768)` | prefill batch size |
| `enable_prefix_caching` | True | shared system prompt cached across items |
| `enable_chunked_prefill` | True | |
| `kv_cache_dtype` | `fp8` | |

To change generation budget: edit `MAX_TOKENS` in §2 only — `max_model_len`, `max_num_seqs`, and the output filename all follow.


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# max_num_seqs heuristic: KV cache per seq grows ~linearly with max_model_len.
# At 17500 the A100 profile ran 256 seqs comfortably; halve for ~34k context.
_max_num_seqs = max(64, int(256 * 17500 / MAX_MODEL_LEN))

with _jupyter_stdout_for_vllm():
    llm = LLM(
        model=MODEL_ID,
        dtype="bfloat16",
        enable_prefix_caching=True,
        gpu_memory_utilization=0.88,
        max_model_len=MAX_MODEL_LEN,
        trust_remote_code=True,
        max_num_seqs=_max_num_seqs,
        max_num_batched_tokens=max(MAX_MODEL_LEN, 32768),
        enable_chunked_prefill=True,
        kv_cache_dtype="fp8",
    )

default_sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print(f"Model loaded.  max_model_len={MAX_MODEL_LEN}  max_num_seqs={_max_num_seqs}  sampling.max_tokens={MAX_TOKENS}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

INFO 06-01 03:13:57 [core.py:109] Initializing a V1 LLM engine (v0.20.0) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=18432, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=fp8, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, co

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

INFO 06-01 03:14:19 [weight_utils.py:615] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 19.052324 seconds
INFO 06-01 03:14:19 [weight_utils.py:904] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 7.49 GiB. Available RAM: 79.87 GiB.
INFO 06-01 03:14:19 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 06-01 03:14:21 [default_loader.py:384] Loading weights took 2.37 seconds
INFO 06-01 03:14:22 [gpu_model_runner.py:4879] Model loading took 7.61 GiB memory and 22.459878 seconds
INFO 06-01 03:14:33 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/00fbac81bd/rank_0_0/backbone for vLLM's torch.compile
INFO 06-01 03:14:33 [backends.py:1128] Dynamo bytecode transform time: 10.26 s
INFO 06-01 03:14:42 [backends.py:376] Cache the graph of compile range (1, 32768) for later use
INFO 06-01 03:14:49 [backends.py:391] Compiling a graph for compile range (1, 32768) takes 15.35 s
INFO 06-01 03:14:55 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/37554f43cb7cdf6db9e5125b8eda275ec12d34a4a9457cbb6ff2d518363561f6/rank_0_0/model
INFO 06-01 03:14:55 [monitor.py:53] torch.compile took 32.71 s in total
INFO 06-01 03:14:55 [monitor.py:81] Initial profiling/warmup run took 0.29 s
INFO 06-01 03:15:06 [gpu_model_run

## 7. Generate responses

Checkpoint: `results/submission.responses.jsonl` (Drive on Colab). Delete checkpoint to regenerate from scratch.

In [7]:
CHUNK_SIZE = 128

try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(SUBMISSION_CSV).parent

_results_dir.mkdir(parents=True, exist_ok=True)
response_checkpoint = _results_dir / (Path(SUBMISSION_CSV).stem + ".responses.jsonl")
print(f"Checkpoint path: {response_checkpoint}")

completed: dict = {}
if response_checkpoint.exists():
    with open(response_checkpoint) as f:
        for line in f:
            r = json.loads(line)
            completed[r["id"]] = r["response"]
    print(f"Resumed from checkpoint: {len(completed)} responses already generated")

remaining = [d for d in data if d.get("id") not in completed]
print(f"Generating {len(remaining)} remaining / {len(data)} total...")

for chunk_start in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]

    prompts = []
    chunk_params = []
    for item in chunk:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)
        chunk_params.append(default_sampling_params)

    with _jupyter_stdout_for_vllm():
        outputs = llm.generate(prompts, sampling_params=chunk_params)

    with open(response_checkpoint, "a") as f:
        for item, out in zip(chunk, outputs):
            response = out.outputs[0].text.strip()
            completed[item["id"]] = response
            f.write(json.dumps({"id": item["id"], "response": response}) + "\n")

    done = len(completed)
    print(f"  {done}/{len(data)}  ({done / len(data) * 100:.1f}%)")

assert len(completed) == len(data), "Missing ids — checkpoint vs DATA_PATH mismatch"
responses = [completed[d["id"]] for d in data]
print(f"Done. {len(responses)} responses ready.")

Rendering prompts:   0%|          | 0/128 [00:00<?, ?it/s]

Checkpoint path: /content/drive/MyDrive/CSE151B/results/submission_exact_v1_16k.responses.jsonl
Generating 943 remaining / 943 total...


Processed prompts:   0%|          | 0/128 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/128 [00:00<?, ?it/s]

  128/943  (13.6%)


...

  768/943  (81.4%)


Processed prompts:   0%|          | 0/128 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/47 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/47 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  896/943  (95.0%)
  943/943  (100.0%)
Done. 943 responses ready.


## 8. Write submission CSV

Uses Python’s `csv` writer so commas and newlines inside `response` are quoted per RFC 4180.

In [8]:
try:
    csv_out = DRIVE_PROJECT_ROOT / "results" / Path(SUBMISSION_CSV).name
except NameError:
    csv_out = Path(SUBMISSION_CSV)

csv_out.parent.mkdir(parents=True, exist_ok=True)

with open(csv_out, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
    w.writerow(["id", "response"])
    for row in data:
        qid = row["id"]
        w.writerow([qid, completed[qid]])

print(f"Wrote {len(data)} rows to {csv_out.resolve()}")

with open(csv_out, newline="", encoding="utf-8") as f:
    n = sum(1 for _ in csv.reader(f))
assert n == len(data) + 1, f"Expected header + {len(data)} rows, got {n} lines"
print("CSV line count OK (1 header +", len(data), "data rows).")

Wrote 943 rows to /content/drive/MyDrive/CSE151B/results/submission_exact_v1_16k.csv
CSV line count OK (1 header + 943 data rows).


## Done

Upload **`submission.csv`** to the course leaderboard workflow. Keep **`submission.responses.jsonl`** as a backup of raw traces.

Pipeline matches **pub-003** (`full_public.ipynb`): 32k tokens, bf16 A100, adaptive multi-blank prompts.

In [9]:
try:
    from google.colab import runtime
    drive.flush_and_unmount()
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")
except NameError:
    print("Drive not mounted — skipping runtime termination.")